# Breast Cancer Diagnosis
## *When Cells Tell a Story*

---

Every year, over 2 million women worldwide are diagnosed with breast cancer. A doctor examines an image of a cell sample under a microscope. The cells look... *off*. But how off? Enough to be cancer?

This is where it gets hard. The difference between a benign lump and a malignant tumor can be subtle — hidden in the shape, size, and texture of individual cells. A wrong call in either direction has consequences: unnecessary surgery, or a missed cancer that spreads.

Today, you have measurements from 569 real cell samples — each described by 30 features computed from a digitized image of a fine needle aspirate. The question is:

> **Can the *shape* of a cell tell you whether it's cancerous?**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="ticks", font_scale=1.15)

df = pd.read_csv("../data/breast_cancer.csv")
malignant = (df["diagnosis"] == 0).sum()
benign    = (df["diagnosis"] == 1).sum()
print(f"Loaded {len(df)} cell samples — {malignant} malignant ({malignant/len(df)*100:.1f}%) · {benign} benign ({benign/len(df)*100:.1f}%)")

In [ ]:
df.sample(8, random_state=42).style.format(precision=3)

## Before You Look at the Data...

Pause for a moment. Each row in this dataset is a real person's biopsy. A doctor is waiting for an answer. A patient is waiting for a call.

*What do you think distinguishes a cancerous cell from a healthy one?* Would it be size? Shape? Texture? How irregular the boundary looks?

Hold that thought. Now let's see if the data agrees with your intuition. Use the controls below to compare how different cell measurements differ between malignant and benign samples. Try different features.

**Which measurements separate the two groups most clearly?**

In [ ]:
nice_names = {col: col.replace("_", " ").title() for col in df.columns if col != "diagnosis"}
mean_features = [c for c in df.columns if c.startswith("mean")]

@widgets.interact(
    feature_x=widgets.Dropdown(options=mean_features, value="mean radius",   description="X axis:"),
    feature_y=widgets.Dropdown(options=mean_features, value="mean concave points", description="Y axis:"),
)
def explore(feature_x, feature_y):
    labels = {0: "Malignant", 1: "Benign"}
    colors = {0: "#E8575A", 1: "#5B8FB9"}

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    counts = df["diagnosis"].value_counts().rename(labels)
    axes[0].pie(counts, labels=counts.index, autopct="%1.1f%%",
                colors=[colors[1], colors[0]], startangle=90,
                wedgeprops={"edgecolor": "white", "linewidth": 2})
    axes[0].set_title(f"Dataset Balance\n{len(df)} samples", weight="bold")

    for val in [0, 1]:
        subset = df[df["diagnosis"] == val]
        axes[1].hist(subset[feature_x], bins=25, alpha=0.55,
                     label=labels[val], color=colors[val])
    axes[1].set_xlabel(nice_names[feature_x])
    axes[1].set_ylabel("Count")
    axes[1].set_title(f"Distribution of {nice_names[feature_x]}", weight="bold")
    axes[1].legend()

    for val in [0, 1]:
        subset = df[df["diagnosis"] == val]
        axes[2].scatter(subset[feature_x], subset[feature_y],
                        alpha=0.5, s=30, label=labels[val], color=colors[val],
                        edgecolors="white", linewidths=0.3)
    axes[2].set_xlabel(nice_names[feature_x])
    axes[2].set_ylabel(nice_names[feature_y])
    axes[2].set_title(f"{nice_names[feature_x]} vs {nice_names[feature_y]}", weight="bold")
    axes[2].legend()

    plt.suptitle(f"Breast Cancer — Exploring Cell Measurements",
                 fontsize=13, weight="bold", y=1.02)
    sns.despine()
    plt.tight_layout()
    plt.show()

## From Intuition to Algorithm

By now, you've probably noticed some clear patterns — perhaps that malignant cells tend to be larger, more irregularly shaped, or have more concave points along their boundary.

You built that understanding by looking at the data yourself: choosing features, comparing distributions, noticing where the groups separate. But what if we could teach a computer to discover these patterns on its own?

That's the idea behind a **decision tree**. It looks at the cell measurements and figures out which questions to ask — *"Is the mean radius above 15?" "Are there more than 0.05 concave points?"* — to best predict whether a sample is malignant or benign. Nobody tells it the answer. It learns from examples.

Here's the twist: we train it on *most* of the data, then test it on samples it has **never seen**. If it still predicts well, it has truly *learned* the pattern — not just memorized the answers. Use the slider to control how many questions the tree is allowed to ask, and watch what happens as it grows more complex.

In [ ]:
selected_features = ["mean radius", "mean texture", "mean perimeter", "mean area",
                     "mean smoothness", "mean compactness", "mean concavity",
                     "mean concave points", "mean symmetry", "mean fractal dimension"]
features_df = df[selected_features].copy()
target = df["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(
    features_df, target, test_size=0.2, random_state=42
)

feature_names = [c.replace("mean ", "").title() for c in selected_features]
class_names   = ["Malignant", "Benign"]

@widgets.interact(depth=widgets.IntSlider(
    value=3, min=1, max=6, step=1,
    description="Tree depth:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
))
def train_and_show(depth):
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train, y_train)
    preds    = clf.predict(X_test)
    accuracy = accuracy_score(y_test, preds)
    cm       = confusion_matrix(y_test, preds)

    fig, axes = plt.subplots(1, 2, figsize=(16, max(4, depth * 1.8)))

    plot_tree(clf, feature_names=feature_names, class_names=class_names,
              filled=True, rounded=True, fontsize=9, ax=axes[0])
    axes[0].set_title(f"Decision Tree (depth = {depth})", weight="bold", fontsize=13)

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1],
                xticklabels=class_names, yticklabels=class_names,
                linewidths=1, linecolor="white")
    axes[1].set_xlabel("Predicted", fontsize=11)
    axes[1].set_ylabel("Actual", fontsize=11)
    axes[1].set_title(f"Confusion Matrix — Accuracy: {accuracy*100:.1f}%",
                      weight="bold", fontsize=13)

    plt.tight_layout()
    plt.show()

    print(f"\nModel accuracy on unseen test data: {accuracy*100:.1f}%")
    print(f"Training set: {len(X_train)} samples | Test set: {len(X_test)} samples")

## What Did You Discover?

Think about the journey you just took. You started with a question — *can the shape of a cell tell you whether it's cancerous?* — and no clear answer.

You formed a hypothesis, explored the data with your own eyes, and found patterns that reveal something remarkable: malignant cells *do* look different — they tend to be larger, more irregularly shaped, and have more concave indentations along their boundaries. These are features a pathologist might notice under a microscope, but now a machine can learn them too.

Then you watched a decision tree learn those same patterns entirely on its own. Nobody told the algorithm "check the concave points" — it discovered that rule from the data, just like you did. Did you notice what happened when the tree got too deep? More complex isn't always better — a model can *memorize* the training data instead of learning the real pattern. In medicine, that difference matters: a model that only works on data it's already seen is useless in the clinic.

That is the heart of data science. Not the code, not the math — but the ability to **ask a meaningful question, explore the evidence, and let the data guide you toward an answer.**

---

*This dataset comes from the University of Wisconsin, where Dr. William H. Wolberg used fine needle aspirates to diagnose breast masses. The same data has helped train models used in real clinical decision support. Today, you've walked the same path.*